# Этап 1: Парсинг и разметка отзывов Wildberries

В этом ноутбуке:
1. Собираем отзывы с товаров продавца
2. Размечаем их на фейковые и реальные по эвристикам

In [ ]:
import sys
sys.path.append('../src')

import json
import pandas as pd
from parser import WBReviewParser
from labeler import ReviewLabeler

## 1.1 Получение nmId товаров

**Инструкция:**
1. Откройте https://www.wildberries.ru/seller/1177893
2. Нажмите F12 (DevTools)
3. Перейдите на вкладку Network
4. Обновите страницу
5. Найдите запросы к API (catalog, products)
6. В ответах найдите поле nmId для товаров
7. Добавьте их в список ниже

In [ ]:
# Замените на реальные nmId товаров продавца 1177893
nm_ids = [
    # Пример: 12345678, 87654321, 11223344
]

print(f"Будем парсить {len(nm_ids)} товаров")

## 1.2 Парсинг отзывов

In [ ]:
parser = WBReviewParser(delay=2.0)

# Собираем отзывы (максимум 200 на товар)
reviews = parser.parse_multiple_products(nm_ids, max_per_product=200)

print(f"Собрано {len(reviews)} отзывов")

In [ ]:
# Сохраняем сырые данные
parser.save_to_json(reviews, '../data/raw/reviews_raw.json')

# Просмотр примера
pd.DataFrame(reviews).head()

## 1.3 Разметка отзывов

Используем эвристические правила:
- **Фейк (1)**: Рейтинг 5 + короткий текст (<150 символов) + 2+ ключевых слова
- **Реальный (0)**: Рейтинг ≤3 ИЛИ содержит конкретику (цифры, время использования)
- **Серый (-1)**: Остальное (исключаем из обучения)

In [ ]:
labeler = ReviewLabeler()
df_labeled = labeler.label_dataset(reviews)

print(f"\nИтоговый датасет: {len(df_labeled)} отзывов")
print(f"Распределение классов:\n{df_labeled['label'].value_counts()}")

In [ ]:
# Примеры фейковых отзывов
print("=== Примеры ФЕЙКОВЫХ отзывов ===")
fake_samples = df_labeled[df_labeled['label'] == 1].sample(min(5, len(df_labeled[df_labeled['label'] == 1])))
for idx, row in fake_samples.iterrows():
    print(f"Рейтинг: {row['rating']}, Текст: {row['text'][:100]}...\n")

In [ ]:
# Примеры реальных отзывов
print("=== Примеры РЕАЛЬНЫХ отзывов ===")
real_samples = df_labeled[df_labeled['label'] == 0].sample(min(5, len(df_labeled[df_labeled['label'] == 0])))
for idx, row in real_samples.iterrows():
    print(f"Рейтинг: {row['rating']}, Текст: {row['text'][:100]}...\n")

In [ ]:
# Сохраняем размеченные данные
labeler.save_labeled_data(df_labeled, '../data/processed/reviews_labeled.json')

print("✓ Данные сохранены в data/processed/reviews_labeled.json")